In [ ]:
# 1. Install deps & Load Environment

import sys
import os
import json
import pandas as pd

In [ ]:
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/nlp'):
        !git clone -b lab-12-branch --single-branch https://github.com/jaYulichka46/nlp.git
    
    %cd /content/nlp
    !pip install pandas numpy scikit-learn matplotlib seaborn -q
    sys.path.append('/content/nlp')

    FOLDER_ID = '1Xhu4xjZpRu-RP730-hyErp5F0C3l_EvO'
    
    os.makedirs('data', exist_ok=True)
    !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O data/
    
    data_dir = 'data/processed_v2'
else:
    sys.path.append(os.path.abspath('..'))
    data_dir = '../data/processed_v2'

In [ ]:
if 'google.colab' in sys.modules:
    # Встановлюємо залежності для локальної Llama-3 (Unsloth)
    !pip install -q transformers accelerate bitsandbytes
    !pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --upgrade --no-cache-dir git+https://github.com/unslothai/unsloth-zoo.git
    !pip install --no-deps xformers trl peft accelerate bitsandbytes -q

# Підключаємо корінь проєкту
project_root = '/content/nlp_uni' if 'google.colab' in sys.modules else os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [ ]:
# 2. Data / test cases

# 10 тестових кейсів українських новин (різні категорії, шум, відсутність сутностей)
TEST_CASES = [
    {"id": "case_01", "text": "Сьогодні президент Зеленський зустрівся з представниками НАТО в Києві.", "expected_behavior": "Категорія: Politics. Сутності: Зеленський, НАТО, Київ."},
    {"id": "case_02", "text": "Сили ППО вночі збили 10 російських дронів над Одесою.", "expected_behavior": "Категорія: War/Military. Сутності: ППО, Одеса."},
    {"id": "case_03", "text": "Курс долара знову зріс, а МВФ обіцяє новий транш для підтримки економіки.", "expected_behavior": "Категорія: Economy. Сутності: МВФ."},
    {"id": "case_04", "text": "Збірна України з футболу виграла вирішальний матч на чемпіонаті.", "expected_behavior": "Категорія: Sports. Сутностей зі словника може не бути."},
    {"id": "case_05", "text": "Завтра очікується сонячна погода без опадів, температура до +20.", "expected_behavior": "Категорія: General/Unknown. Сутностей немає."},
    {"id": "case_06", "text": "Байден і Макрон провели телефонну розмову щодо нових санкцій.", "expected_behavior": "Категорія: Politics. Сутності: Байден, Макрон."},
    {"id": "case_07", "text": "ШОК! Всі ціни злетять в космос вже завтра!!! Терміново купуйте гречку!", "expected_behavior": "Категорія: Economy (через слово 'ціни'). Noisy text."},
    {"id": "case_08", "text": "Верховна Рада ухвалила новий закон про бюджет для підтримки ЗСУ.", "expected_behavior": "Конфлікт категорій (Politics vs War vs Economy). Інструмент має визначити найрелевантнішу. Сутності: Верховна Рада, ЗСУ."},
    {"id": "case_09", "text": "Сирський відвідав передові позиції на Донеччині.", "expected_behavior": "Категорія: War/Military. Сутності: Сирський, Донеччина."},
    {"id": "case_10", "text": "Кабмін виділив додаткові кошти на відновлення Харкова.", "expected_behavior": "Категорія: Politics/Economy. Сутності: Кабмін, Харків."}
]

print(f"Завантажено {len(TEST_CASES)} тестових кейсів новин.")

In [ ]:
# 3. Tool definitions (Testing)

from src.tools import extract_key_entities, categorize_news

sample_news = {"text": "Сьогодні в Києві Кабмін обговорив співпрацю з МВФ."}

print("Entity Extraction:", extract_key_entities(sample_news))
print("News Categorization:", categorize_news(sample_news))

In [ ]:
# 4. Tool call logger

from src.tool_logger import ToolLogger

LOG_FILE = os.path.join(project_root, "docs", "tool_logs_lab12.jsonl")

# Очищуємо файл логів перед новим запуском тестів
os.makedirs(os.path.dirname(LOG_FILE), exist_ok=True)
with open(LOG_FILE, "w", encoding="utf-8") as f:
    pass

logger = ToolLogger(log_path=LOG_FILE)
print(f"Логер готовий: {LOG_FILE}")

In [ ]:
# 5. Agent design & LLM Setup

import torch
from unsloth import FastLanguageModel
from src.agent import ToolGroundedAgent

print("Завантаження локальної LLM (Unsloth/Llama-3)... Зачекайте.")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True
)
FastLanguageModel.for_inference(model)

def local_llm_caller(prompt):
    """Обгортка для виклику локальної моделі"""
    messages = [
        {"role": "system", "content": "You are a precise JSON-only agent."},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True, temperature=0.0, do_sample=False)
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()

# Ініціалізуємо Агента
agent = ToolGroundedAgent(llm_caller=local_llm_caller, logger=logger, max_steps=4)

In [ ]:
# 6. Baseline LLM without tools

sample_input = TEST_CASES[0]['text']
baseline_prompt = f"Проаналізуй цю українську новину. Визнач її категорію та витягни ключові сутності (люди, організації, локації). Новина: '{sample_input}'"

print(f"News: {sample_input}")
print(f"Baseline (Тільки LLM): {local_llm_caller(baseline_prompt)}")

In [ ]:
# 7. Agent with tools

# Тестуємо агента з інструментами на тому ж прикладі
print(f"News: {sample_input}")
agent_response = agent.run(task_id='test_run_01', user_input=sample_input)
print(f"Agent (Tools + LLM): {json.dumps(agent_response, indent=2, ensure_ascii=False)}")

In [ ]:
# 8. Run 10 test cases

from src.eval_agent import run_evaluation

# Очищуємо логи перед фінальним прогоном
with open(LOG_FILE, "w", encoding="utf-8") as f:
    pass

# Проганяємо всі 10 кейсів
print("Запуск оцінки (це займе кілька хвилин)...")
evaluation_results = run_evaluation(TEST_CASES, agent, baseline_llm_caller=local_llm_caller)
print("Оцінку завершено!")

In [ ]:
# 9. Tool call logs

import json

print("Останні збережені логи викликів інструментів:")
with open(LOG_FILE, "r", encoding="utf-8") as f:
    logs = [json.loads(line) for line in f]

for log in logs[-5:]: # Виводимо останні 5 для компактності
    print(json.dumps(log, indent=2, ensure_ascii=False))

In [ ]:
# 10. Metrics

import json
from collections import defaultdict

with open(LOG_FILE, "r", encoding="utf-8") as f:
    logs = [json.loads(line) for line in f]

# Автоматичні метрики
total_calls = len(logs)
success_calls = sum(1 for log in logs if log.get("success"))
error_calls = total_calls - success_calls

success_rate = (success_calls / total_calls * 100) if total_calls > 0 else 0
error_rate = (error_calls / total_calls * 100) if total_calls > 0 else 0

total_tasks = len(TEST_CASES)
calls_per_task = total_calls / total_tasks

logs_by_task = defaultdict(list)
for log in logs:
    logs_by_task[log["task_id"]].append(log)

tasks_with_multiple_tools = sum(1 for task_logs in logs_by_task.values() if len(task_logs) >= 2)
percent_multiple_tools = (tasks_with_multiple_tools / total_tasks) * 100

# Ручні метрики
manual_metrics = {
    "tasks_with_useful_tool_use": 8,
    "unnecessary_tool_call_count": 2,
    "correct_answers": 7,
    "partly_correct_answers": 3,
    "wrong_answers": 0,               
    "tasks_tool_ignored": 0,          
    "tasks_contradicts_tool": 0       
}

percent_ignored = (manual_metrics["tasks_tool_ignored"] / total_tasks) * 100
percent_contradicts = (manual_metrics["tasks_contradicts_tool"] / total_tasks) * 100

print("Метрики Агента")
print(f"1. Tool call success rate: {success_rate:.1f}%")
print(f"2. Average tool calls per task: {calls_per_task:.1f}")
print(f"Tool error rate: {error_rate:.1f}%")
print(f"% задач, де agent використав >= 2 tools: {percent_multiple_tools:.1f}%")
print(f"3. Tasks with useful tool use: {manual_metrics['tasks_with_useful_tool_use']}")
print(f"4. Unnecessary tool call count: {manual_metrics['unnecessary_tool_call_count']}")
print("5. Final answer correctness:")
print(f"correct: {manual_metrics['correct_answers']}")
print(f"partly correct: {manual_metrics['partly_correct_answers']}")
print(f"wrong: {manual_metrics['wrong_answers']}")

# 11. Error analysis

Аналіз показових та проблемних прикладів роботи Tool-grounded агента. Головне спостереження: агент дуже добре справляється з класифікацією, але обмежений хардкодним словником для сутностей.

1. **Task ID: case_01 (Зеленський, НАТО, Київ)**
* *Expected behavior:* Politics. Сутності: Зеленський, НАТО, Київ.
* *Actual tool calls:* `categorize_news`, `extract_key_entities`.
* *Final answer:* Агент чітко повернув усі 3 сутності та категорію.
* *Error category:* Немає помилки (Ідеальний кейс).

2. **Task ID: case_05 (Прогноз погоди)**
* *Expected behavior:* General/Unknown, відсутність політичних сутностей.
* *Actual tool calls:* Агент викликав обидва інструменти.
* *Final answer:* Агент констатував, що категорія Unknown і сутностей не знайдено.
* *Error category:* Unnecessary tool call (зайвий виклик екстракції сутностей для прогнозу погоди). 
* *Possible fix:* Змусити агента спочатку викликати `categorize_news`, і якщо це "General/Unknown", пропускати пошук сутностей.

3. **Task ID: case_04 (Збірна України з футболу)**
* *Expected behavior:* Sports. Але наш `extract_key_entities` не має футболістів у словнику.
* *Actual tool calls:* Обидва інструменти.
* *Error category:* Tool limitation / Partly correct. Агент правильно визначив категорію (Спорт), але не знайшов сутностей, оскільки інструмент шукає лише політиків та ЗСУ. Baseline модель (без інструментів) тут спрацювала б краще, самостійно витягнувши слово "Україна".

4. **Task ID: case_08 (Закон про бюджет для ЗСУ)**
* *Expected behavior:* Визначити одну з домінантних категорій.
* *Actual tool calls:* `categorize_news` (повернув Politics, бо слів "Верховна Рада", "закон" більше, ніж "ЗСУ").
* ** Категорія Politics, сутності: Верховна Рада, ЗСУ.
* *Error category:* Немає помилки. Демонстрація того, як правило більшості в інструменті вирішує конфлікти категорій, позбавляючи LLM від галюцинацій.

5. **Task ID: case_07 (Клікбейт про ціни)**
* *Actual tool calls:* `categorize_news` (помилково зачепився за слово "ціни" і видав Economy).
* *Error category:* Tool output incorrect.
* *Possible fix:* Додати в інструмент перевірку на клікбейтні патерни (капслок, багато знаків оклику), щоб категорія була "Spam".

In [ ]:
# 12. Generate docs/audit_summary_lab12.md

summary_content = """# Audit Summary Lab 12 - Tool-grounded Agent

1. **Use case:** News Analyst Assistant (Аналіз українських новин).
2. **Tools:** `extract_key_entities` (rule-based пошук сутностей), `categorize_news` (keyword-based категоризація).
3. **Test cases:** 10 (включаючи політику, війну, спорт, клікбейт та прогноз погоди).
4. **Tool call success rate:** 100.0% (Функції Python відпрацювали без помилок).
5. **Average tool calls per task:** ~2.0 (Агент системно викликав обидва інструменти).
6. **Корисність:** Агент успішно структурував відповіді. Завдяки `categorize_news`, категорія новини завжди була строго з нашого списку (War, Politics, Economy, Sports), тоді як Baseline LLM часто вигадувала власні формати відповідей або писала довгі есе замість чіткого фактажу.
7. **Зайві виклики (Unnecessary calls):** Зафіксовано для прогнозу погоди та клікбейту. Агент шукав політичні сутності там, де їх очевидно не могло бути.
8. **Найкращі приклади:**
   - `case_01` та `case_09`: Агент блискавично класифікував новини та витягнув усіх потрібних осіб (Зеленський, Сирський) та локації, спираючись на JSON від інструментів.
9. **Проблемні приклади:**
   - `case_04` (Спорт): Жорсткий словник в інструменті екстракції не дозволив знайти специфічні спортивні сутності. Тут інструмент обмежив можливості LLM.
10. **Що б ви покращували далі:**
   - Реалізувати послідовну логіку (Chain of Thought): спочатку визначати категорію, і лише якщо це не "General/Unknown", викликати екстрактор сутностей.
   - Замінити хардкодний словник в `extract_key_entities` на виклик справжньої NER моделі (наприклад, української моделі Stanza або SpaCy).
"""

os.makedirs(os.path.dirname(LOG_FILE), exist_ok=True)
with open(os.path.join(project_root, "docs", "audit_summary_lab12.md"), "w", encoding="utf-8") as f:
    f.write(summary_content)

print("Файл docs/audit_summary_lab12.md згенеровано.")